# Day 4 — Explicit Planning & Controlled Execution: Plan Before You Act

**Course:** Agentic Systems for Forward Deployed Engineers  
**Theme:** Giving an LLM direct access to side-effects is unsafe. Day 4 inserts a validation firewall between *what the LLM wants to do* and *what the system allows it to do*. The LLM generates a plan; deterministic Python validates and executes it.

---

## What this day covers

| Layer | Skill |
|---|---|
| Risk assessment | `derive_policy_overlay` — deterministic risk flags from case facts |
| Explicit planning | LLM generates `ExecutionPlan` as structured JSON; schema-validated before execution |
| Plan validation | 15+ safety checks run before a single task fires |
| Controlled execution | Serial task executor with stop conditions, dependency resolution, ledger |
| Recommendation gate | Final gated decision — 9 boolean conditions must all pass |
| Adversarial exit check | The £48,000 + missing PO + bank change scenario |

## The core insight for FDEs

> *"The planner proposes. The validator disposes. The executor runs what survives."*

Separating **planning** from **execution** creates an audit firewall:  
- The plan is a document — it can be inspected, rejected, and stored before any action is taken.  
- The executor only runs tasks the plan declared — it cannot improvise.  
- Stop conditions halt the executor deterministically — no LLM decision required.

---

## Prerequisites

```bash
pip install -r requirements.txt
```
Planning sections require an `.env` file with Azure OpenAI credentials.  
Policy overlay, validation, and execution sections work fully offline.

In [ ]:
import sys, json
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

print("✓ repo root:", repo_root)

---
## Part 1 — Case Facts: The Planner's Input

`CaseFacts` is the typed input to the Day 4 planning system.  
It is a structured summary of the invoice context — not the raw invoice itself.  
In production this would be assembled from Day 1–3 outputs.

The Day 4 system ships with four fixture scenarios that test increasingly complex risk combinations:

In [ ]:
from aegisap.day4.planning.plan_types import CaseFacts

# Load fixture case facts from day4 fixtures
fixture_dir = repo_root / "fixtures" / "day4"
fixtures = sorted(fixture_dir.glob("*.json"))

print(f"Day 4 fixture cases:\n")
for fpath in fixtures:
    data = json.loads(fpath.read_text())
    # case_facts is nested inside the fixture
    cf_data = data.get("case_facts", data)
    print(f"  {fpath.stem}:")
    print(f"    invoice_amount_gbp:   {cf_data.get('invoice_amount_gbp')}")
    print(f"    po_present:           {cf_data.get('po_present')}")
    print(f"    bank_details_changed: {cf_data.get('bank_details_changed')}")
    print(f"    supplier_exists:      {cf_data.get('supplier_exists')}")
    print()

---
## Part 2 — The Policy Overlay: Deterministic Risk Flags

`derive_policy_overlay` is called **before** the LLM sees the case.  
It reads `CaseFacts` and deterministically derives:
- Which **risk flags** apply
- Which **task types** are mandatory
- Which **stop conditions** would halt execution
- Which **approvals** are required

This means the planner cannot forget a mandatory check — the validator will catch any omission.

In [ ]:
from aegisap.day4.planning.plan_types import CaseFacts
from aegisap.day4.planning.policy_overlay import derive_policy_overlay

# Define four cases showing escalating risk
case_scenarios = [
    ("clean_path", CaseFacts(
        case_id="C-001", invoice_id="INV-001", supplier_id="SUP-001",
        supplier_name="Contoso Ltd", supplier_exists=True,
        invoice_amount_gbp=5_000.00, invoice_currency="GBP",
        po_present=True, po_number="PO-001",
        bank_details_changed=False,
    )),
    ("missing_po", CaseFacts(
        case_id="C-002", invoice_id="INV-002", supplier_id="SUP-001",
        supplier_name="Contoso Ltd", supplier_exists=True,
        invoice_amount_gbp=5_000.00, invoice_currency="GBP",
        po_present=False,
        bank_details_changed=False,
    )),
    ("high_value", CaseFacts(
        case_id="C-003", invoice_id="INV-003", supplier_id="SUP-001",
        supplier_name="Contoso Ltd", supplier_exists=True,
        invoice_amount_gbp=48_000.00, invoice_currency="GBP",
        po_present=True, po_number="PO-003",
        bank_details_changed=False,
    )),
    ("combined_risk", CaseFacts(
        case_id="C-004", invoice_id="INV-004", supplier_id="SUP-001",
        supplier_name="Contoso Ltd", supplier_exists=True,
        invoice_amount_gbp=48_000.00, invoice_currency="GBP",
        po_present=False,
        bank_details_changed=True, bank_change_verified=False,
    )),
]

for name, case_facts in case_scenarios:
    overlay = derive_policy_overlay(case_facts)
    print(f"\n{'='*50}")
    print(f"Scenario: {name}")
    print(f"  risk_flags:          {overlay.risk_flags}")
    print(f"  mandatory_tasks:     {overlay.mandatory_task_types}")
    print(f"  stop_conditions:     {overlay.global_stop_conditions}")
    print(f"  required_approvals:  {overlay.required_approvals}")
    print(f"  recommendation_blocks: {overlay.recommendation_blocks}")

> **Key insight:** The `combined_risk` scenario (high value + missing PO + bank change) requires **7 mandatory task types** and triggers 6 stop conditions.  
> An LLM planner that omits even one of these will fail validation — before any task runs.

---
## Part 3 — The Execution Plan Schema

The LLM is asked to produce a JSON object matching `ExecutionPlan`.  
Let's inspect the schema to understand what a valid plan looks like.

In [ ]:
from aegisap.day4.planning.plan_types import ExecutionPlan, PlanTask, RecommendationGate, PLAN_TASK_TYPES

print("ExecutionPlan top-level fields:")
for name, field in ExecutionPlan.model_fields.items():
    print(f"  {name:<40} {str(field.annotation)[:60]}")

print(f"\nAvailable task types ({len(PLAN_TASK_TYPES)}):")
for t in PLAN_TASK_TYPES:
    print(f"  {t}")

print("\nRecommendationGate conditions:")
for name in RecommendationGate.model_fields:
    print(f"  {name}")

In [ ]:
from aegisap.day4.planning.plan_types import (
    ExecutionPlan, PlanTask, RecommendationGate, ActionClassification, EscalationRoute
)

clean_plan = ExecutionPlan(
    plan_id='plan-clean-001', case_id='C-001',
    goal='Process clean-path invoice and draft payment recommendation',
    case_risk_flags=['clean_path', 'existing_supplier'],
    planning_rationale='No risk flags triggered. Standard mandatory tasks only.',
    tasks=[
        PlanTask(
            task_id='T-01', task_type='policy_retrieval', owner_agent='policy_retriever',
            depends_on=[], required_evidence=[],
            expected_outputs=['policy_requirements', 'citation_refs', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
        PlanTask(
            task_id='T-02', task_type='vendor_history_check', owner_agent='vendor_history_verifier',
            depends_on=['T-01'], required_evidence=[],
            expected_outputs=['supplier_history_summary', 'risk_level', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
        PlanTask(
            task_id='T-03', task_type='payment_recommendation_draft', owner_agent='payment_recommendation_agent',
            depends_on=['T-01', 'T-02'], required_evidence=[],
            expected_outputs=['recommendation_draft', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
    ],
    global_preconditions=[], global_stop_conditions=[], escalation_triggers=[],
    escalation_route=EscalationRoute(queue='finance_ops_manual_review', owners=['ap_manager']),
    escalation_reason_template='Escalate for manual review when mandatory controls remain unresolved.',
    required_approvals=[],
    recommendation_gate=RecommendationGate(
        all_mandatory_tasks_completed=False, required_evidence_present=False,
        no_unresolved_blockers=False, confidence_thresholds_met=False,
        required_approvals_identified=False,
        mandatory_escalations_resolved_or_triggered=False,
        po_condition_satisfied=True, bank_change_verification_satisfied=True,
        approval_path_satisfied=True,
    ),
    action_classification=ActionClassification(
        current_stage='reversible', irreversible_actions_allowed=False,
    ),
)

print(f'Plan ID:      {clean_plan.plan_id}')
print(f'Tasks:        {len(clean_plan.tasks)}')
print(f'Risk flags:   {clean_plan.case_risk_flags}')
print(f'Task types:   {[t.task_type for t in clean_plan.tasks]}')

---
## Part 4 — Plan Validation: The Safety Firewall

`validate_execution_plan` runs **before** any task executes.  
It has 15+ checks covering:
- Schema correctness (Pydantic)
- Duplicate task IDs
- Dependency graph cycles
- Mandatory task types present (from overlay)
- Required preconditions and stop conditions declared
- Escalation triggers and approvals included
- `irreversible_actions_allowed` must be `false` at plan time
- High-risk tasks (`vendor_bank_verification`) must escalate on failure

In [ ]:
from aegisap.day4.planning.plan_validator import validate_execution_plan

# Validate the clean_path plan against its overlay
_, clean_overlay = next((n, derive_policy_overlay(cf)) for n, cf in case_scenarios if n == "clean_path")

result = validate_execution_plan(clean_plan, clean_overlay)
print(f"Valid: {result.valid}")
if result.errors:
    for e in result.errors:
        print(f"  Error: {e}")
else:
    print("  No validation errors ✓")

In [ ]:
# Now take the combined_risk overlay and try to validate the clean_path plan against it
# The validator should catch all the missing mandatory tasks, stop conditions, triggers, and approvals
from aegisap.day4.planning.policy_overlay import derive_policy_overlay

_, combined_overlay = next((n, derive_policy_overlay(cf)) for n, cf in case_scenarios if n == "combined_risk")

result = validate_execution_plan(clean_plan, combined_overlay)
print(f"Valid: {result.valid}")
print(f"\nValidation errors ({len(result.errors)}):")
for i, e in enumerate(result.errors, 1):
    print(f"  [{i}] {e}")

> **This is the firewall in action.** The LLM-generated plan was correct for a clean-path case,  
> but it could **never** be used for a combined-risk case — even if the LLM tried.  
> The validator is deterministic, fast, and runs before anything writes to any system.

---
## Part 5 — The Task Registry and Worker Contracts

Each task type in the plan maps to a registered **worker**.  
Workers receive a `WorkerTaskInput` and return a `TaskResult`.  
The executor validates the result against the task contract before accepting it — a worker cannot return partial or malformed data.

In [ ]:
from aegisap.day4.execution.task_registry import TaskRegistry
from aegisap.day4.execution.default_workers import default_workers

registry = TaskRegistry()
workers = default_workers()
for w in workers:
    registry.register(w)

print('Registered task workers:')
for w in workers:
    print(f'  {w.task_type:<40} → {type(w).__name__}')

In [ ]:
from aegisap.day4.execution.task_contracts import get_task_contract

# Inspect the contract for the highest-risk task
contract = get_task_contract('vendor_bank_verification')
print(f'Task type:              {contract.task_type}')
print(f'Description:            {contract.description}')
print(f'Expected output fields: {contract.expected_output_fields}')
print(f'Owner agent:            {contract.owner_agent}')

---
## Part 6 — The Execution Engine

The executor walks the task dependency graph.  
In each iteration it finds **ready tasks** — those whose `depends_on` tasks are all completed.  
It runs one task at a time (serial for inspectability), validates the result, updates the ledger, and checks stop conditions.

```
while pending tasks:
    find ready tasks (deps all completed)
    if none → mark remaining blocked, return
    execute next ready task
    validate result against contract
    update ledger entry
    check global stop conditions
    if stop condition met → mark remaining skipped, return
```

In [ ]:
# Run the executor offline with correct task contracts.
import asyncio
from aegisap.day4.planning.plan_types import (
    CaseFacts, ExecutionPlan, PlanTask, RecommendationGate, EscalationRoute, ActionClassification
)
from aegisap.day4.planning.policy_overlay import derive_policy_overlay
from aegisap.day4.planning.plan_validator import validate_execution_plan
from aegisap.day4.state.workflow_state import create_initial_workflow_state
from aegisap.day4.execution.task_registry import TaskRegistry
from aegisap.day4.execution.default_workers import default_workers
from aegisap.day4.execution.plan_executor import execute_plan

cf = CaseFacts(
    case_id='C-RUN-001', invoice_id='INV-001', supplier_id='SUP-001',
    supplier_name='Contoso Ltd', supplier_exists=True,
    invoice_amount_gbp=4_800.00, invoice_currency='GBP',
    po_present=True, po_number='PO-001', bank_details_changed=False,
)
overlay = derive_policy_overlay(cf)
print(f'Case: {cf.case_id}  amount=£{cf.invoice_amount_gbp:,.0f}')
print(f'risk_flags:      {overlay.risk_flags}')
from aegisap.day4.planning.plan_types import (
    ExecutionPlan, PlanTask, RecommendationGate, ActionClassification, EscalationRoute
)

plan = ExecutionPlan(
    plan_id='plan-run-001', case_id='C-RUN-001',
    goal='Clean-path invoice: policy check then draft recommendation.',
    case_risk_flags=list(overlay.risk_flags),
    planning_rationale='No elevated risk flags.',
    tasks=[
        PlanTask(
            task_id='T-01', task_type='policy_retrieval', owner_agent='policy_retriever',
            depends_on=[], required_evidence=[],
            expected_outputs=['policy_requirements', 'citation_refs', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
        PlanTask(
            task_id='T-02', task_type='vendor_history_check', owner_agent='vendor_history_verifier',
            depends_on=['T-01'], required_evidence=[],
            expected_outputs=['supplier_history_summary', 'risk_level', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
        PlanTask(
            task_id='T-03', task_type='payment_recommendation_draft', owner_agent='payment_recommendation_agent',
            depends_on=['T-01', 'T-02'], required_evidence=[],
            expected_outputs=['recommendation_draft', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
    ],
    global_preconditions=[], global_stop_conditions=[], escalation_triggers=[],
    escalation_route=EscalationRoute(queue='finance_ops_manual_review', owners=['ap_manager']),
    escalation_reason_template='Escalate when controls unresolved.',
    required_approvals=[],
    recommendation_gate=RecommendationGate(
        all_mandatory_tasks_completed=False, required_evidence_present=False,
        no_unresolved_blockers=False, confidence_thresholds_met=False,
        required_approvals_identified=False,
        mandatory_escalations_resolved_or_triggered=False,
        po_condition_satisfied=True, bank_change_verification_satisfied=True,
        approval_path_satisfied=True,
    ),
    action_classification=ActionClassification(
        current_stage='reversible', irreversible_actions_allowed=False,
    ),
)

vr = validate_execution_plan(plan, overlay)
print(f'\nPlan valid: {vr.valid}')
if vr.errors:
    for e in vr.errors: print(f'  error: {e}')

state = create_initial_workflow_state(cf)
registry = TaskRegistry()
for w in default_workers():
    registry.register(w)

state = asyncio.run(execute_plan(state=state, plan=plan, registry=registry))
print(f'\nExecution complete:')
print(f'  plan_status:   {state.planning.plan_status}')
print(f'  escalation:    {state.planning.escalation_status}')
print(f'  ledger:        {[(e.task_id, e.status) for e in state.task_ledger]}')

---
## Part 7 — The Recommendation Gate

After execution, the `RecommendationGate` determines whether a payment recommendation can be issued.  
All 9 boolean conditions must be `True` before a recommendation is composed.  
If any condition is `False`, the case escalates instead.

This is the final safety layer — it ensures the system never recommends payment on an unresolved risk.

In [ ]:
# evaluate_recommendation_gate takes (WorkflowState, ExecutionPlan).
# Build a minimal plan inline so this works without a live LLM.
from aegisap.day4.recommendation.recommendation_gate import evaluate_recommendation_gate
from aegisap.day4.planning.plan_types import (
    CaseFacts, TaskLedgerEntry, ExecutionPlan, PlanTask,
    RecommendationGate, EscalationRoute, ActionClassification
)
from aegisap.day4.state.workflow_state import create_initial_workflow_state

cf_gate = CaseFacts(
    case_id='C-GATE-001', invoice_id='INV-001', supplier_id='SUP-001',
    supplier_name='Contoso Ltd', supplier_exists=True,
    invoice_amount_gbp=5_000.00, invoice_currency='GBP',
    po_present=True, po_number='PO-001', bank_details_changed=False,
)
from aegisap.day4.planning.plan_types import (
    ExecutionPlan, PlanTask, RecommendationGate, ActionClassification, EscalationRoute
)

gate_plan = ExecutionPlan(
    plan_id='plan-gate-001', case_id='C-GATE-001',
    goal='Clean path: verify policy and draft recommendation',
    case_risk_flags=['clean_path', 'existing_supplier'],
    planning_rationale='Standard mandatory tasks only.',
    tasks=[
        PlanTask(
            task_id='T-01', task_type='policy_retrieval', owner_agent='policy_retriever',
            depends_on=[], required_evidence=[],
            expected_outputs=['policy_requirements', 'citation_refs', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
        PlanTask(
            task_id='T-02', task_type='payment_recommendation_draft', owner_agent='payment_recommendation_agent',
            depends_on=['T-01'], required_evidence=[],
            expected_outputs=['recommendation_draft', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
    ],
    global_preconditions=[], global_stop_conditions=[], escalation_triggers=[],
    escalation_route=EscalationRoute(queue='finance_ops_manual_review', owners=['ap_manager']),
    escalation_reason_template='Escalate when controls unresolved.',
    required_approvals=[],
    recommendation_gate=RecommendationGate(
        all_mandatory_tasks_completed=False, required_evidence_present=False,
        no_unresolved_blockers=False, confidence_thresholds_met=False,
        required_approvals_identified=False,
        mandatory_escalations_resolved_or_triggered=False,
        po_condition_satisfied=True, bank_change_verification_satisfied=True,
        approval_path_satisfied=True,
    ),
    action_classification=ActionClassification(
        current_stage='reversible', irreversible_actions_allowed=False
    ),
)

# State A: all tasks completed → should be eligible
passing_state = create_initial_workflow_state(cf_gate)
passing_state.task_ledger = [
    TaskLedgerEntry(task_id=t.task_id, task_type=t.task_type, owner_agent="agent",
                    depends_on=[], status='completed', confidence=0.95, outputs={})
    for t in gate_plan.tasks
]

# State B: blocked by stop conditions
blocked_state = create_initial_workflow_state(cf_gate)
blocked_state.eligibility.blocking_conditions = ['stop_condition_met: unresolved_blocker']

for label, state in [('All tasks completed', passing_state), ('Blocked state', blocked_state)]:
    result = evaluate_recommendation_gate(state, gate_plan)
    print(f'\nScenario: {label}')
    print(f'  eligible: {result.eligible}')
    for r in result.reasons:
        print(f'  reason:   {r}')

---
## Part 8 — The Adversarial Exit Check: £48,000 + Missing PO + Bank Change

This is the scenario from `docs/DAY_04_EXIT_CHECK.md` — the hardest test case.  

**The setup:**
- Existing supplier (known vendor)
- Invoice for £48,000 GBP (above £25k approval threshold and £40k controller threshold)
- No PO reference attached
- Bank account changed (unverified)

**The expected system behaviour:**
1. Policy overlay fires `combined_risk` flag + 7 mandatory task types
2. Any plan omitting mandatory tasks is rejected before execution
3. If the executor runs, the `unverified_bank_change` stop condition halts it mid-run
4. The recommendation gate finds at minimum 3 failing conditions
5. The system escalates — it does **not** recommend payment

In [ ]:
from aegisap.day4.planning.plan_types import CaseFacts
from aegisap.day4.planning.policy_overlay import derive_policy_overlay

adversarial_case = CaseFacts(
    case_id="ADVERSARIAL-001",
    invoice_id="INV-ADV-001",
    supplier_id="SUP-001",
    supplier_name="Contoso Ltd",
    supplier_exists=True,
    invoice_amount_gbp=48_000.00,
    invoice_currency="GBP",
    po_present=False,
    bank_details_changed=True,
    bank_change_verified=False,
    amount_approval_threshold_gbp=25_000.00,
    controller_approval_threshold_gbp=40_000.00,
)

overlay = derive_policy_overlay(adversarial_case)

print("=== Adversarial Case Policy Overlay ===")
print(f"Risk flags:             {overlay.risk_flags}")
print(f"Mandatory tasks:        {overlay.mandatory_task_types}")
print(f"Stop conditions:        {overlay.global_stop_conditions}")
print(f"Required approvals:     {overlay.required_approvals}")
print(f"Escalation owners:      {overlay.escalation_route.owners}")
print(f"Recommendation blocks:  {overlay.recommendation_blocks}")
print(f"\nEscalation reason template:\n  {overlay.escalation_reason_template}")

In [ ]:
# Build an adversarial combined-risk plan inline and evaluate the gate.
from aegisap.day4.recommendation.recommendation_gate import evaluate_recommendation_gate
from aegisap.day4.planning.plan_types import (
    CaseFacts, TaskLedgerEntry, ExecutionPlan, PlanTask,
    RecommendationGate, EscalationRoute, ActionClassification
)
from aegisap.day4.state.workflow_state import create_initial_workflow_state

# adversarial_case was defined in Part 8 setup cell above
from aegisap.day4.planning.plan_types import (
    ExecutionPlan, PlanTask, RecommendationGate, ActionClassification, EscalationRoute
)

adv_plan = ExecutionPlan(
    plan_id='plan-adv-001', case_id='ADVERSARIAL-001',
    goal='Process combined-risk invoice',
    case_risk_flags=['combined_risk','missing_po','bank_details_changed','high_value','existing_supplier'],
    planning_rationale='All risk flags active. Maximum mandatory controls.',
    tasks=[
        PlanTask(
            task_id='T-01', task_type='policy_retrieval', owner_agent='policy_retriever',
            depends_on=[], required_evidence=[],
            expected_outputs=['policy_requirements', 'citation_refs', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
        PlanTask(
            task_id='T-02', task_type='po_match_verification', owner_agent='po_match_agent',
            depends_on=['T-01'], required_evidence=[],
            expected_outputs=['po_match_status', 'matched_fields', 'missing_fields', 'mismatch_flags', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
        PlanTask(
            task_id='T-03', task_type='po_waiver_check', owner_agent='po_waiver_verifier',
            depends_on=['T-02'], required_evidence=[],
            expected_outputs=['waiver_present', 'waiver_authority', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
        PlanTask(
            task_id='T-04', task_type='vendor_bank_verification', owner_agent='vendor_risk_verifier',
            depends_on=['T-01'], required_evidence=[],
            expected_outputs=['bank_change_verified', 'authoritative_source_checked', 'risk_level', 'confidence'],
            preconditions=[], stop_if_missing=True, min_confidence=0.7,
            on_failure='escalate', action_class='reversible', inputs={},
        ),
        PlanTask(
            task_id='T-05', task_type='threshold_approval_check', owner_agent='approval_policy_agent',
            depends_on=['T-01'], required_evidence=[],
            expected_outputs=['required_approvals', 'approval_path_defined', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
        PlanTask(
            task_id='T-06', task_type='vendor_history_check', owner_agent='vendor_history_verifier',
            depends_on=['T-01'], required_evidence=[],
            expected_outputs=['supplier_history_summary', 'risk_level', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
        PlanTask(
            task_id='T-07', task_type='manual_escalation_package', owner_agent='escalation_composer',
            depends_on=['T-02', 'T-03', 'T-04', 'T-05', 'T-06'], required_evidence=[],
            expected_outputs=['escalation_reasons', 'evidence_bundle', 'confidence'],
            preconditions=[], stop_if_missing=False, min_confidence=0.7,
            on_failure='block', action_class='reversible', inputs={},
        ),
    ],
    global_preconditions=['po_verified_or_waived','bank_change_authoritatively_verified',
                          'approval_route_defined_for_threshold_case'],
    global_stop_conditions=['missing_required_po_evidence','unverified_bank_change',
                            'combined_risk_requires_manual_review'],
    escalation_triggers=['vendor_bank_verification_failed'],
    escalation_route=EscalationRoute(queue='finance_ops_manual_review',
                                    owners=['ap_manager','controller']),
    escalation_reason_template='Combined risk flags require manual review.',
    required_approvals=['ap_manager','controller'],
    recommendation_gate=RecommendationGate(
        all_mandatory_tasks_completed=False, required_evidence_present=False,
        no_unresolved_blockers=False, confidence_thresholds_met=False,
        required_approvals_identified=True,
        mandatory_escalations_resolved_or_triggered=True,
        po_condition_satisfied=False, bank_change_verification_satisfied=False,
        approval_path_satisfied=False,
    ),
    action_classification=ActionClassification(
        current_stage='reversible', irreversible_actions_allowed=False,
    ),
)

# Simulate partial execution halted by stop conditions
adv_state = create_initial_workflow_state(adversarial_case)
adv_state.task_ledger = [
    TaskLedgerEntry(task_id='T-01', task_type='policy_retrieval', owner_agent='policy_retriever',
                    depends_on=[], status='completed', confidence=0.9, outputs={}),
]
adv_state.eligibility.blocking_conditions = [
    'stop_condition_met: unverified_bank_change',
    'stop_condition_met: missing_required_po_evidence',
]
adv_state.eligibility.missing_evidence = ['bank_verification_record', 'po_match_or_waiver']
adv_state.planning.escalation_status = 'triggered'

result = evaluate_recommendation_gate(adv_state, adv_plan)
print(f'eligible: {result.eligible}')
print(f'\nBlocking reasons ({len(result.reasons)}):')
for r in result.reasons:
    print(f'  - {r}')
print('\n→ System will ESCALATE, not recommend payment ✓')

---
## Part 9 — Running All Four Fixtures

The fixture files in `fixtures/day4/` include pre-built execution plans so the full flow can be tested without a live LLM.  
Let's validate each plan against its overlay to confirm correctness.

In [ ]:
from aegisap.day4.planning.plan_types import CaseFacts
from aegisap.day4.planning.policy_overlay import derive_policy_overlay
import json
from pathlib import Path

fixture_dir = repo_root / 'fixtures' / 'day4'

print(f'{'Fixture':<40} {'risk_flags':<45} mandatory_tasks')
print('-' * 110)

for fpath in sorted(fixture_dir.glob('*.json')):
    data = json.loads(fpath.read_text())
    cf_data = data.get('case_facts', {})
    try:
        case_facts = CaseFacts.model_validate(cf_data)
        overlay = derive_policy_overlay(case_facts)
        flags_str = str(overlay.risk_flags)[:43]
        tasks_str = str(overlay.mandatory_task_types)[:50]
        print(f'{fpath.stem:<40} {flags_str:<45} {len(overlay.mandatory_task_types)} tasks')
    except Exception as e:
        print(f'{fpath.stem:<40} ERROR: {e}')

---
## Exercises

### Exercise 1 — Add a new risk flag: currency mismatch

Day 4 doesn't yet handle detection of a currency mismatch (an invoice in USD from a GBP supplier).  

1. Add a `"currency_mismatch"` case to `RiskFlag` (as a local extension here — no need to edit source).
2. Write a `derive_currency_mismatch` function that takes `CaseFacts` and returns `True` when `invoice_currency != "GBP"`.
3. Specify what mandatory tasks and stop conditions it should add.
4. Write a test showing the policy overlay fires for a USD invoice.

In [ ]:
# Exercise 1 workspace
from aegisap.day4.planning.plan_types import CaseFacts
from aegisap.day4.planning.policy_overlay import derive_policy_overlay

# USD invoice from a GBP supplier
usd_case = CaseFacts(
    case_id="USD-001",
    invoice_id="INV-USD-001",
    supplier_id="SUP-001",
    supplier_name="Contoso Ltd",
    supplier_exists=True,
    invoice_amount_gbp=5_000.00,
    invoice_currency="USD",  # ← mismatch
    po_present=True,
    po_number="PO-USD-001",
    bank_details_changed=False,
)

# Current overlay (no currency mismatch detection yet)
overlay = derive_policy_overlay(usd_case)
print("Current risk flags:", overlay.risk_flags)
print()
print("Your task: extend this to detect 'currency_mismatch' for USD invoices")

### Exercise 2 — Build an invalid plan and watch validation catch it

Create an `ExecutionPlan` for the `missing_po` scenario that has the following deliberate errors:
1. A task that depends on a non-existent `task_id`.
2. Missing the mandatory `po_match_verification` task type.
3. `irreversible_actions_allowed: true` — an explicit safety violation.

Then run `validate_execution_plan` and verify all three errors are caught.

In [ ]:
# Exercise 2 workspace
from aegisap.day4.planning.plan_validator import validate_execution_plan
from aegisap.day4.planning.policy_overlay import derive_policy_overlay
from aegisap.day4.planning.plan_types import CaseFacts

missing_po_case = CaseFacts(
    case_id="C-002", invoice_id="INV-002", supplier_id="SUP-001",
    supplier_name="Contoso Ltd", supplier_exists=True,
    invoice_amount_gbp=5_000.00, invoice_currency="GBP",
    po_present=False, bank_details_changed=False,
)
overlay = derive_policy_overlay(missing_po_case)
print("Mandatory task types:", overlay.mandatory_task_types)
print()
print("Build your invalid plan here and run validate_execution_plan against the overlay")

### Exercise 3 — Trace a task dependency chain

Given the `high_value_missing_po_bank_change` fixture plan:
1. Load the plan and draw the dependency graph (text or a simple dict representation).
2. Identify which task will be the *last to run*.
3. Which task, if it fails, would block the most downstream tasks?

In [ ]:
# Exercise 3 workspace
from aegisap.day4.planning.plan_schema import parse_execution_plan
import json
from pathlib import Path

fixture_path = repo_root / "fixtures" / "day4" / "high_value_missing_po_bank_change.json"

if fixture_path.exists():
    data = json.loads(fixture_path.read_text())
    plan_data = data.get("execution_plan")

    if plan_data:
        plan = parse_execution_plan(plan_data)
        print("Task dependency graph:")
        for task in plan.tasks:
            deps = task.depends_on if task.depends_on else ["(start)"]
            print(f"  {task.task_id} ({task.task_type}) ← depends on: {deps}")
    else:
        print("No execution_plan found in fixture")
else:
    print("Fixture not found. Manually build a dependency chain for your chosen scenario.")

---
## Summary — Day 4 Principles

| Principle | Implementation |
|---|---|
| Plan before you act | `ExecutionPlan` is a validated JSON document — inspected, stored, before execution |
| Deterministic overlay | `derive_policy_overlay` fires risk flags from case facts — LLM cannot override |
| Validation firewall | `validate_execution_plan` runs 15+ checks before any task executes |
| Typed task contracts | Workers receive and return structured types — no free-form outputs |
| Stop conditions | Executor halts deterministically on risk conditions — no LLM decision needed |
| Recommendation gate | 9 boolean conditions must all pass before a payment recommendation is composed |
| Adversarial resilience | The exit-check scenario cannot produce a payment recommendation by design |

## What comes next (Day 5+)

Days 1–4 have built the **data foundation** (Day 1), **workflow engine** (Day 2), **multi-agent retrieval** (Day 3), and **explicit planning with guardrails** (Day 4).

The next phase of the course will cover:
- **Day 5:** Azure infrastructure deployment — provisioning and wiring the services you've been mocking
- **Day 6:** Production observability — distributed tracing, cost dashboards, alerting
- **Day 7:** Human-in-the-loop workflows — approval queues, escalation handling, audit logs
- **Day 8–10:** Full production rollout, CI/CD, red-teaming, and operating at scale